# Feedforward Neural Network - Experiments

Notebook ini berisi eksperimen-eksperimen sesuai spesifikasi Tugas Besar 1 IF3270 Pembelajaran Mesin.

## 1. Setup dan Import Library

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import sys
import os

# Add src to path
sys.path.insert(0, os.path.join(os.getcwd(), '..'))

# Import modules
from src.models.ffnn import FFNN
from src.preprocessing import DataPreprocessor
from src.utils.metrics import accuracy, precision, recall, f1_score, confusion_matrix
from src.utils.plotting import plot_training_history, plot_weight_distribution, plot_gradient_distribution
from src.utils.io import save_training_history, load_training_history

# Set style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

print("Library imported successfully!")

## 2. Preprocessing Dataset

In [ ]:
# Path ke dataset
data_path = 'data/datasetml_2026.csv'

# Preprocessing
preprocessor = DataPreprocessor(data_path)
preprocessor.load_data()
preprocessor.explore_data()
preprocessor.visualize_data(save_path='data/eda_visualization.png')

# Preprocess dan split data
X_train, X_val, X_test, y_train, y_val, y_test = preprocessor.preprocess_data(
    test_size=0.2,
    val_size=0.2,
    random_state=42
)

# Dapatkan info data
info = preprocessor.get_data_info()
print(f"\nInformasi Dataset:")
print(f"  Jumlah fitur: {info['n_features']}")
print(f"  Jumlah kelas: {info['n_classes']}")
print(f"  Train samples: {info['n_train_samples']}")
print(f"  Val samples: {info['n_val_samples']}")
print(f"  Test samples: {info['n_test_samples']}")

## 3. Eksperimen 1: Pengaruh Depth dan Width

Menganalisis pengaruh banyak layer (depth) dan banyak neuron per layer (width)

In [ ]:
# Konfigurasi eksperimen
n_features = info['n_features']
n_classes = info['n_classes']

# 3 variasi width (depth tetap = 3 layer)
width_configs = [
    {'name': 'Narrow (8 neurons)', 'layer_sizes': [n_features, 8, 8, n_classes]},
    {'name': 'Medium (16 neurons)', 'layer_sizes': [n_features, 16, 16, n_classes]},
    {'name': 'Wide (32 neurons)', 'layer_sizes': [n_features, 32, 32, n_classes]}
]

width_results = {}
epochs = 50

for config in width_configs:
    print(f"\n{'='*60}")
    print(f"Testing: {config['name']}")
    print(f"{'='*60}")
    
    model = FFNN(
        layer_sizes=config['layer_sizes'],
        activations=['relu', 'relu', 'softmax'],
        loss_function='categorical_cross_entropy',
        initializer='uniform',
        learning_rate=0.01
    )
    
    history = model.train(
        X_train, y_train, X_val, y_val,
        batch_size=32,
        learning_rate=0.01,
        epochs=epochs,
        verbose=1
    )
    
    # Evaluasi
    y_pred = model.predict(X_test)
    acc = accuracy(y_test, y_pred)
    
    width_results[config['name']] = {
        'model': model,
        'history': history,
        'accuracy': acc,
        'final_train_loss': history['train_loss'][-1],
        'final_val_loss': history['val_loss'][-1]
    }
    
    print(f"Test Accuracy: {acc:.4f}")

In [ ]:
# 3 variasi depth (width tetap = 16 neurons)
depth_configs = [
    {'name': 'Shallow (2 hidden)', 'layer_sizes': [n_features, 16, 16, n_classes]},
    {'name': 'Medium (3 hidden)', 'layer_sizes': [n_features, 16, 16, 16, n_classes]},
    {'name': 'Deep (4 hidden)', 'layer_sizes': [n_features, 16, 16, 16, 16, n_classes]}
]

depth_results = {}

for config in depth_configs:
    print(f"\n{'='*60}")
    print(f"Testing: {config['name']}")
    print(f"{'='*60}")
    
    activations = ['relu'] * (len(config['layer_sizes']) - 2) + ['softmax']
    
    model = FFNN(
        layer_sizes=config['layer_sizes'],
        activations=activations,
        loss_function='categorical_cross_entropy',
        initializer='uniform',
        learning_rate=0.01
    )
    
    history = model.train(
        X_train, y_train, X_val, y_val,
        batch_size=32,
        learning_rate=0.01,
        epochs=epochs,
        verbose=1
    )
    
    # Evaluasi
    y_pred = model.predict(X_test)
    acc = accuracy(y_test, y_pred)
    
    depth_results[config['name']] = {
        'model': model,
        'history': history,
        'accuracy': acc,
        'final_train_loss': history['train_loss'][-1],
        'final_val_loss': history['val_loss'][-1]
    }
    
    print(f"Test Accuracy: {acc:.4f}")

In [ ]:
# Plot perbandingan width
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

# Width comparison - Training Loss
for i, (name, result) in enumerate(width_results.items()):
    axes[0, i].plot(result['history']['train_loss'], label='Train Loss')
    axes[0, i].plot(result['history']['val_loss'], label='Val Loss')
    axes[0, i].set_title(f'{name}\nAcc: {result["accuracy"]:.4f}')
    axes[0, i].set_xlabel('Epoch')
    axes[0, i].set_ylabel('Loss')
    axes[0, i].legend()
    axes[0, i].grid(True, alpha=0.3)

# Depth comparison - Training Loss
for i, (name, result) in enumerate(depth_results.items()):
    axes[1, i].plot(result['history']['train_loss'], label='Train Loss')
    axes[1, i].plot(result['history']['val_loss'], label='Val Loss')
    axes[1, i].set_title(f'{name}\nAcc: {result["accuracy"]:.4f}')
    axes[1, i].set_xlabel('Epoch')
    axes[1, i].set_ylabel('Loss')
    axes[1, i].legend()
    axes[1, i].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('data/experiment1_depth_width.png', dpi=300, bbox_inches='tight')
plt.show()

print("\nHasil Eksperimen 1 (Depth & Width):")
print("\nWidth Variations:")
for name, result in width_results.items():
    print(f"  {name}: Acc={result['accuracy']:.4f}, Train Loss={result['final_train_loss']:.4f}, Val Loss={result['final_val_loss']:.4f}")

print("\nDepth Variations:")
for name, result in depth_results.items():
    print(f"  {name}: Acc={result['accuracy']:.4f}, Train Loss={result['final_train_loss']:.4f}, Val Loss={result['final_val_loss']:.4f}")

## 4. Eksperimen 2: Pengaruh Fungsi Aktivasi

Menganalisis pengaruh berbagai fungsi aktivasi pada hidden layer

In [ ]:
# Base arsitektur
base_layer_sizes = [n_features, 32, 16, n_classes]  # 3 layer

# Fungsi aktivasi untuk diuji (kecuali softmax untuk output)
activations_to_test = ['linear', 'relu', 'sigmoid', 'tanh']

activation_results = {}

for activation in activations_to_test:
    print(f"\n{'='*60}")
    print(f"Testing: {activation.upper()} activation")
    print(f"{'='*60}")
    
    model = FFNN(
        layer_sizes=base_layer_sizes,
        activations=[activation, activation, 'softmax'],
        loss_function='categorical_cross_entropy',
        initializer='uniform',
        learning_rate=0.01
    )
    
    history = model.train(
        X_train, y_train, X_val, y_val,
        batch_size=32,
        learning_rate=0.01,
        epochs=epochs,
        verbose=1
    )
    
    # Evaluasi
    y_pred = model.predict(X_test)
    acc = accuracy(y_test, y_pred)
    
    activation_results[activation] = {
        'model': model,
        'history': history,
        'accuracy': acc,
        'final_train_loss': history['train_loss'][-1],
        'final_val_loss': history['val_loss'][-1]
    }
    
    print(f"Test Accuracy: {acc:.4f}")

In [ ]:
# Plot perbandingan aktivasi
fig, axes = plt.subplots(2, 2, figsize=(15, 10))
axes = axes.flatten()

for i, (activation, result) in enumerate(activation_results.items()):
    axes[i].plot(result['history']['train_loss'], label='Train Loss', linewidth=2)
    axes[i].plot(result['history']['val_loss'], label='Val Loss', linewidth=2)
    axes[i].set_title(f'{activation.upper()} - Acc: {result["accuracy"]:.4f}')
    axes[i].set_xlabel('Epoch')
    axes[i].set_ylabel('Loss')
    axes[i].legend()
    axes[i].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('data/experiment2_activations.png', dpi=300, bbox_inches='tight')
plt.show()

# Plot distribusi bobot untuk beberapa aktivasi
fig, axes = plt.subplots(2, 2, figsize=(15, 10))
axes = axes.flatten()

for i, (activation, result) in enumerate(activation_results.items()):
    model = result['model']
    weights_flat = np.concatenate([w.flatten() for w in model.weights])
    axes[i].hist(weights_flat, bins=50, alpha=0.7, edgecolor='black')
    axes[i].set_title(f'{activation.upper()} - Weight Distribution')
    axes[i].set_xlabel('Weight Value')
    axes[i].set_ylabel('Frequency')
    axes[i].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('data/experiment2_weight_distribution.png', dpi=300, bbox_inches='tight')
plt.show()

print("\nHasil Eksperimen 2 (Fungsi Aktivasi):")
for activation, result in activation_results.items():
    print(f"  {activation}: Acc={result['accuracy']:.4f}, Train Loss={result['final_train_loss']:.4f}, Val Loss={result['final_val_loss']:.4f}")

## 5. Eksperimen 3: Pengaruh Learning Rate

Menganalisis pengaruh learning rate terhadap training

In [ ]:
# Konfigurasi learning rates
learning_rates = [0.001, 0.01, 0.1]

lr_results = {}

for lr in learning_rates:
    print(f"\n{'='*60}")
    print(f"Testing: Learning Rate = {lr}")
    print(f"{'='*60}")
    
    model = FFNN(
        layer_sizes=[n_features, 32, 16, n_classes],
        activations=['relu', 'relu', 'softmax'],
        loss_function='categorical_cross_entropy',
        initializer='uniform',
        learning_rate=lr
    )
    
    history = model.train(
        X_train, y_train, X_val, y_val,
        batch_size=32,
        learning_rate=lr,
        epochs=epochs,
        verbose=1
    )
    
    # Evaluasi
    y_pred = model.predict(X_test)
    acc = accuracy(y_test, y_pred)
    
    lr_results[lr] = {
        'model': model,
        'history': history,
        'accuracy': acc,
        'final_train_loss': history['train_loss'][-1],
        'final_val_loss': history['val_loss'][-1]
    }
    
    print(f"Test Accuracy: {acc:.4f}")

In [ ]:
# Plot perbandingan learning rate
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for i, (lr, result) in enumerate(lr_results.items()):
    axes[i].plot(result['history']['train_loss'], label='Train Loss', linewidth=2)
    axes[i].plot(result['history']['val_loss'], label='Val Loss', linewidth=2)
    axes[i].set_title(f'LR={lr} - Acc: {result["accuracy"]:.4f}')
    axes[i].set_xlabel('Epoch')
    axes[i].set_ylabel('Loss')
    axes[i].legend()
    axes[i].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('data/experiment3_learning_rate.png', dpi=300, bbox_inches='tight')
plt.show()

# Plot distribusi gradien
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for i, (lr, result) in enumerate(lr_results.items()):
    model = result['model']
    if model.weight_gradients:
        grad_flat = np.concatenate([g.flatten() for g in model.weight_gradients])
        axes[i].hist(grad_flat, bins=50, alpha=0.7, color='orange', edgecolor='black')
        axes[i].set_title(f'LR={lr} - Gradient Distribution')
        axes[i].set_xlabel('Gradient Value')
        axes[i].set_ylabel('Frequency')
        axes[i].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('data/experiment3_gradient_distribution.png', dpi=300, bbox_inches='tight')
plt.show()

print("\nHasil Eksperimen 3 (Learning Rate):")
for lr, result in lr_results.items():
    print(f"  LR={lr}: Acc={result['accuracy']:.4f}, Train Loss={result['final_train_loss']:.4f}, Val Loss={result['final_val_loss']:.4f}")

## 6. Eksperimen 4: Pengaruh Regularisasi

Membandingkan tanpa regularisasi, L1, dan L2

In [ ]:
# Konfigurasi regularisasi
regularization_configs = [
    {'name': 'No Regularization', 'regularizer': None},
    {'name': 'L1 (lambda=0.01)', 'regularizer': {'type': 'l1', 'lambda_param': 0.01}},
    {'name': 'L2 (lambda=0.01)', 'regularizer': {'type': 'l2', 'lambda_param': 0.01}}
]

reg_results = {}

for config in regularization_configs:
    print(f"\n{'='*60}")
    print(f"Testing: {config['name']}")
    print(f"{'='*60}")
    
    model = FFNN(
        layer_sizes=[n_features, 32, 16, n_classes],
        activations=['relu', 'relu', 'softmax'],
        loss_function='categorical_cross_entropy',
        initializer='uniform',
        learning_rate=0.01,
        regularizer=config['regularizer']
    )
    
    history = model.train(
        X_train, y_train, X_val, y_val,
        batch_size=32,
        learning_rate=0.01,
        epochs=epochs,
        verbose=1
    )
    
    # Evaluasi
    y_pred = model.predict(X_test)
    acc = accuracy(y_test, y_pred)
    
    reg_results[config['name']] = {
        'model': model,
        'history': history,
        'accuracy': acc,
        'final_train_loss': history['train_loss'][-1],
        'final_val_loss': history['val_loss'][-1]
    }
    
    print(f"Test Accuracy: {acc:.4f}")

In [ ]:
# Plot perbandingan regularisasi
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for i, (name, result) in enumerate(reg_results.items()):
    axes[i].plot(result['history']['train_loss'], label='Train Loss', linewidth=2)
    axes[i].plot(result['history']['val_loss'], label='Val Loss', linewidth=2)
    axes[i].set_title(f'{name}\nAcc: {result["accuracy"]:.4f}')
    axes[i].set_xlabel('Epoch')
    axes[i].set_ylabel('Loss')
    axes[i].legend()
    axes[i].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('data/experiment4_regularization.png', dpi=300, bbox_inches='tight')
plt.show()

print("\nHasil Eksperimen 4 (Regularisasi):")
for name, result in reg_results.items():
    print(f"  {name}: Acc={result['accuracy']:.4f}, Train Loss={result['final_train_loss']:.4f}, Val Loss={result['final_val_loss']:.4f}")

## 7. Eksperimen 5: Perbandingan dengan Sklearn MLP

In [ ]:
from sklearn.neural_network import MLPClassifier

# Train model kita
print("Training FFNN Implementation...")
our_model = FFNN(
    layer_sizes=[n_features, 32, 16, n_classes],
    activations=['relu', 'relu', 'softmax'],
    loss_function='categorical_cross_entropy',
    initializer='uniform',
    learning_rate=0.01
)

our_history = our_model.train(
    X_train, y_train, X_val, y_val,
    batch_size=32,
    learning_rate=0.01,
    epochs=epochs,
    verbose=1
)

our_pred = our_model.predict(X_test)
our_acc = accuracy(y_test, our_pred)

# Train sklearn MLP
print("\nTraining Sklearn MLP...")
sklearn_model = MLPClassifier(
    hidden_layer_sizes=(32, 16),
    activation='relu',
    learning_rate_init=0.01,
    max_iter=epochs,
    batch_size=32,
    random_state=42,
    verbose=True
)

# One-hot encode y_train untuk sklearn
y_train_classes = np.argmax(y_train, axis=1) if y_train.ndim > 1 else y_train
y_test_classes = np.argmax(y_test, axis=1) if y_test.ndim > 1 else y_test

sklearn_model.fit(X_train, y_train_classes)
sklearn_pred = sklearn_model.predict(X_test)
sklearn_acc = accuracy(y_test_classes, sklearn_pred)

print(f"\n{'='*60}")
print(f"HASIL PERBANDINGAN")
print(f"{'='*60}")
print(f"FFNN Implementation: {our_acc:.4f}")
print(f"Sklearn MLP:         {sklearn_acc:.4f}")
print(f"\nSelisih: {abs(our_acc - sklearn_acc):.4f}")

## 8. Kesimpulan

Berdasarkan eksperimen yang telah dilakukan, dapat disimpulkan:

1. **Depth dan Width**: [analisis hasil]
2. **Fungsi Aktivasi**: [analisis hasil]
3. **Learning Rate**: [analisis hasil]
4. **Regularisasi**: [analisis hasil]
5. **Perbandingan dengan Sklearn**: [analisis hasil]